# Week 4 L1 — Activity 1: Clean two small datasets

A community health service has sent you two spreadsheets exported as CSV files:

| File | What it holds |
|---|---|
| `community_health_activity_messy.csv` | appointments and waiting times, per area, year, age group and sex |
| `area_population_messy.csv` | population, region and deprivation band, per area and year |

Both files are messy in the ways real exports usually are: a title line above the column names, inconsistent spelling and capitals, blank cells, numbers typed as words, duplicate rows and impossible values.

By the end of this notebook you will have saved two clean files:

| Clean file | What it holds |
|---|---|
| `community_health_activity_clean.csv` | one row per area × year × age group × sex |
| `area_population_clean.csv` | one row per area × year |

**How to work through it.** Run each cell in order with **Shift + Enter**. Before you run a cell, read the text above it and guess what will happen. After it runs, check whether you were right.

## The five steps of cleaning

1. **Look** at the file before deciding how to read it.
2. **Decide what one row means** (this is called the *grain*).
3. **Fix** the problems one at a time: column names, text, missing values, types, duplicates, impossible values.
4. **Check** that the fixes worked.
5. **Save** the clean result.

We will do every step slowly for the first file, then repeat them more quickly for the second.

## SetUp

Python on its own does not know how to work with tables. We **import** libraries that add that ability.

- `pandas` is the main library for tables. A table in pandas is called a **DataFrame**. We give it the short name `pd`.
- `numpy` adds numerical tools. We only need one thing from it today: `np.nan`, which means "missing number". We give it the short name `np`.
- `Path` (from `pathlib`) builds file paths that work on Windows, Mac and Linux.

In [107]:
%pip install pandas numpy

from pathlib import Path
import pandas as pd
import numpy as np

print("pandas version:", pd.__version__)

Note: you may need to restart the kernel to use updated packages.
pandas version: 3.0.6



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Next we store the two file names in variables, so we only type them once. `Path(".")` means "the folder this notebook is in", and `/` joins a folder and a file name.

> Keep both CSV files in the same folder as this notebook.

In [108]:
DATA_DIR = Path("data")
ACTIVITY_FILE = DATA_DIR / "community_health_activity_messy.csv"
POPULATION_FILE = DATA_DIR / "area_population_messy.csv"

print(ACTIVITY_FILE, "exists:", ACTIVITY_FILE.exists())
print(POPULATION_FILE, "exists:", POPULATION_FILE.exists())

data\community_health_activity_messy.csv exists: True
data\area_population_messy.csv exists: True


Both lines should say `exists: True`. If one says `False`, the file is not in the same folder as the notebook, or its name is spelled differently.

---
# Part 1 — The health activity file

## Step 1: Look before you read

`pd.read_csv()` reads a CSV file into a DataFrame. Normally it assumes the **first line holds the column names**. We should not trust that until we have looked.

Two settings help us peek safely:

- `header=None` — "don't treat any line as column names, just show me the raw lines"
- `nrows=4` — "only read the first 4 lines"

In [109]:
pd.read_csv(ACTIVITY_FILE, header=None, nrows=4)

,0,1,2,3,4,5,6,7,8
0,Community health activity extract - teaching file,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Area Code,AREA NAME,Reporting Year,Age Group,SEX,Appointments,Average Wait Days,Follow-Up Needed,Notes
2,CH001,Northside,2024,0 - 17,F,"1,250",7 days,Yes,school clinics
3,CH001,Northside,2024,18 to 64,Male,980,5,no,NaN


**What do you notice?** Line `0` is a title, not column names. The real column names are on line `1`.

So when we read the file properly we tell pandas to skip the first line with `skiprows=1`.

In [110]:
activity_raw = pd.read_csv(ACTIVITY_FILE, skiprows=1)
activity_raw

,Area Code,AREA NAME,Reporting Year,Age Group,SEX,Appointments,Average Wait Days,Follow-Up Needed,Notes
0,CH001,Northside,2024,0 - 17,F,"1,250",7 days,Yes,school clinics
1,CH001,Northside,2024,18 to 64,Male,980,5,no,NaN
2,CH001,NORTHSIDE,2024,65 plus,Female,430,NaN,Y,telephone follow-up
3,CH002,Riverside,2024,0-17,female,875,6,N,NaN
4,CH002,Riverside,2024,18-64,M,1 040,8 days,yes,evening service
5,CH002,Riverside,2024,65+,Male,310,-1,No,data-entry error in wait
6,CH003,Hilltown,2024,0–17,F,620,9,YES,NaN
7,CH003,Hilltown,2024,18 to 64,male,nine hundred,10,n,NaN
8,CH003,NaN,2024,65 plus,Female,275,12,Yes,area name missing
9,CH003,NaN,2024,65 plus,Female,275,12,Yes,area name missing


We call this `activity_raw` and **never change it**. Every fix happens on a copy, so we can always go back and compare.

## First look: four questions to ask of any table

### 1. How big is it?
`.shape` gives *(number of rows, number of columns)*.

In [111]:
activity_raw.shape

(13, 9)

### 2. What type is each column?
`.dtypes` shows how pandas has stored each column.

- `int64` — whole numbers
- `float64` — numbers with decimals (pandas also uses this when a number column has blanks)
- `object` — text, or a mixture of things pandas could not decide on

A column that should be numbers but shows `object` is a warning sign: something inside it is not a number.

In [112]:
activity_raw.dtypes

Area Code               str
AREA NAME               str
Reporting Year        int64
Age Group               str
SEX                     str
Appointments            str
Average Wait Days       str
Follow-Up Needed        str
Notes                   str
dtype: object

### 3. How many values are missing in each column?
`.isna()` marks every missing cell as `True`. `.sum()` then counts the `True`s in each column (Python treats `True` as 1 and `False` as 0).

In [113]:
activity_raw.isna().sum()

Area Code             0
AREA NAME             2
Reporting Year        0
Age Group             0
SEX                   1
Appointments          0
Average Wait Days     1
Follow-Up Needed      1
Notes                 5
dtype: int64

### 4. Are any rows repeated exactly?
`.duplicated()` marks a row `True` if an identical row appeared above it. `.sum()` counts them.

In [114]:
print(activity_raw.duplicated().sum())

1


Write down what you found. These numbers are your **before** picture; at the end you will compare them with the **after** picture.

> **Try it.** Look at the table above. Which columns contain numbers that pandas has stored as text? Can you see why?

## Wrapping repeated work in a function

We will ask those same four questions again later, for the second file. Instead of retyping them, we put them inside a **function**.

A function is a named, reusable block of code:

```python
def function_name(input_1, input_2):
    # the steps, indented
    ...
```

- `def` starts the definition.
- The names in brackets are the **inputs** (called *parameters*). Inside the function they stand for whatever you pass in.
- The indented lines are the steps.
- Nothing happens when you *define* a function. It only runs when you *call* it: `function_name(something, something_else)`.

This function takes a table `df` and a label `name`, and prints the four checks. `display()` shows a table neatly inside a notebook.

In [115]:
def quick_audit(df, name):
    print(name)
    print("Rows and columns:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    print("Type and missing values per column:")
    summary = pd.DataFrame({"type": df.dtypes, "missing": df.isna().sum()})
    display(summary)

Running that cell printed nothing, which is correct: we have only *defined* the function. Now we call it.

In [116]:
quick_audit(activity_raw, "Raw activity data")

Raw activity data
Rows and columns: (13, 9)
Duplicate rows: 1
Type and missing values per column:


,type,missing
Area Code,str,0
AREA NAME,str,2
Reporting Year,int64,0
Age Group,str,0
SEX,str,1
Appointments,str,0
Average Wait Days,str,1
Follow-Up Needed,str,1
Notes,str,5


## Step 2: Tidy the column names

Look at the column names exactly as they are in the file. `.columns` lists them.

In [117]:
activity_raw.columns

Index([' Area Code ', ' AREA NAME ', ' Reporting Year ', ' Age Group ',
       ' SEX ', ' Appointments ', ' Average Wait Days ', ' Follow-Up Needed ',
       ' Notes '],
      dtype='str')

They contain spaces at the ends, capital letters, brackets and question marks. Names like that are hard to type and easy to get wrong. We want simple names like `area_code`: lower case, words joined by `_`.

We will fix them one small step at a time. `.str` lets us apply a text operation to **every** name at once.

**Step a — remove spaces from both ends** with `.str.strip()`:

In [118]:
names = activity_raw.columns
names = names.str.strip()
names

Index(['Area Code', 'AREA NAME', 'Reporting Year', 'Age Group', 'SEX',
       'Appointments', 'Average Wait Days', 'Follow-Up Needed', 'Notes'],
      dtype='str')

**Step b — make everything lower case** with `.str.lower()`:

In [119]:
names = names.str.lower()
names

Index(['area code', 'area name', 'reporting year', 'age group', 'sex',
       'appointments', 'average wait days', 'follow-up needed', 'notes'],
      dtype='str')

**Step c — replace anything that is not a letter or a digit with `_`.**

`.str.replace()` can take a *pattern* instead of exact text. The pattern `[^a-z0-9]+` reads as:

- `[a-z0-9]` — one lower-case letter or digit
- `^` inside the square brackets — **not** one of those
- `+` — one or more in a row

So a name such as `"average wait (days)"` becomes `"average_wait_days_"`: the space and `(` together become one `_`, and so does the `)` at the end. `regex=True` tells pandas the first argument is a pattern.

In [120]:
names = names.str.replace(r"[^a-z0-9]+", "_", regex=True)
names

Index(['area_code', 'area_name', 'reporting_year', 'age_group', 'sex',
       'appointments', 'average_wait_days', 'follow_up_needed', 'notes'],
      dtype='str')

**Step d — remove any `_` left at either end** with `.str.strip("_")`:

In [121]:
names = names.str.strip("_")
names

Index(['area_code', 'area_name', 'reporting_year', 'age_group', 'sex',
       'appointments', 'average_wait_days', 'follow_up_needed', 'notes'],
      dtype='str')

Those four steps always go together, so we wrap them in a function. Each step hands its result to the next, which is why the function repeats `names = ...`. At the end, `return` sends the finished names back to whoever called the function.

In [122]:
def clean_headers(columns):
    names = columns.str.strip()
    names = names.str.lower()
    names = names.str.replace(r"[^a-z0-9]+", "_", regex=True)
    names = names.str.strip("_")
    return names

Now we make a working copy of the raw table and give it the tidy names. `.copy()` matters: without it, changes to `activity` could also change `activity_raw`.

In [123]:
activity = activity_raw.copy()
activity.columns = clean_headers(activity.columns)
activity.columns

Index(['area_code', 'area_name', 'reporting_year', 'age_group', 'sex',
       'appointments', 'average_wait_days', 'follow_up_needed', 'notes'],
      dtype='str')

## Step 3: Decide what one row means (the grain)

Before fixing values, decide what a single row is supposed to describe. Here, one row should be:

> **one area × one reporting year × one age group × one sex**

So the four columns `area_code`, `reporting_year`, `age_group` and `sex` together should be **unique**: no two rows should share all four. This combination is the table's **key**. We will use it as a check at the end.

> **Try it.** If two rows had the same area, year, age group and sex but different appointment counts, what would that tell you?

## Step 4: Fix the text columns

### 4a. Remove stray spaces

`" Northside"` and `"Northside"` look the same to us but are different to the computer. We strip spaces from every text column.

Two things to notice in the code:

- A **`for` loop** repeats the indented line once for each name in the list `text_columns`. On each pass, `column` holds the next name.
- `.astype("string")` turns the column into pandas' text type. We use it rather than `str` because it keeps blank cells as *missing* (`<NA>`). Using `str` would turn them into the word `"nan"`, which looks like real text.

In [124]:
text_columns = ["area_code", "area_name", "age_group", "sex", "follow_up_needed", "notes"]

for column in text_columns:
    activity[column] = activity[column].astype("string").str.strip()

activity[text_columns]

,area_code,area_name,age_group,sex,follow_up_needed,notes
0,CH001,Northside,0 - 17,F,Yes,school clinics
1,CH001,Northside,18 to 64,Male,no,<NA>
2,CH001,NORTHSIDE,65 plus,Female,Y,telephone follow-up
3,CH002,Riverside,0-17,female,N,<NA>
4,CH002,Riverside,18-64,M,yes,evening service
5,CH002,Riverside,65+,Male,No,data-entry error in wait
6,CH003,Hilltown,0–17,F,YES,<NA>
7,CH003,Hilltown,18 to 64,male,n,<NA>
8,CH003,<NA>,65 plus,Female,Yes,area name missing
9,CH003,<NA>,65 plus,Female,Yes,area name missing


### 4b. Make capitals consistent

`.value_counts()` lists every different value in a column and how often it appears. It is the fastest way to spot spelling variants. `dropna=False` makes it count blanks too.

In [125]:
activity["area_code"].value_counts(dropna=False)

area_code
CH003    5
CH001    4
CH002    4
Name: count, dtype: Int64

Area codes should be all capitals (`.str.upper()`); area names should have a capital first letter (`.str.title()`).

In [126]:
activity["area_code"] = activity["area_code"].str.upper()
activity["area_name"] = activity["area_name"].str.title()

activity["area_code"].value_counts(dropna=False)

area_code
CH003    5
CH001    4
CH002    4
Name: count, dtype: Int64

### 4c. Make age groups consistent

Check the variants first:

In [127]:
activity["age_group"].value_counts(dropna=False)

age_group
65 plus     3
18 to 64    2
0-17        2
18-64       2
65+         2
0 - 17      1
0–17        1
Name: count, dtype: Int64

The same three groups are written in several ways. We want exactly `0-17`, `18-64` and `65+`. Three replacements, one at a time:

1. Replace the long dash `–` with a normal hyphen `-`.
2. Replace the word `to` or a hyphen, with any spaces around it, by a plain `-`. In the pattern `\s*(?:to|-)\s*`: `\s*` means "any number of spaces", and `(?:to|-)` means "`to` or `-`".
3. Replace `65 plus` or `65 +` by `65+`.

`regex=False` on the first line means "this is exact text, not a pattern".

In [128]:
activity["age_group"] = activity["age_group"].str.replace("–", "-", regex=False)
activity["age_group"] = activity["age_group"].str.replace(r"\s*(?:to|-)\s*", "-", regex=True)
activity["age_group"] = activity["age_group"].str.replace(r"65\s*(?:plus|\+)", "65+", regex=True)

activity["age_group"].value_counts(dropna=False)

age_group
65+      5
0-17     4
18-64    4
Name: count, dtype: Int64

### 4d. Map short codes to full words

For `sex`, the file uses `F`, `female`, `M`, `Male` and so on. We lower-case everything first, so there are fewer variants, then use a **dictionary** to translate.

A dictionary is written `{key: value, key: value}`. `.replace(dictionary)` swaps every *key* it finds for its *value*.

In [129]:
activity["sex"].value_counts(dropna=False)

sex
Female    4
F         2
Male      2
M         2
female    1
male      1
<NA>      1
Name: count, dtype: Int64

In [130]:
sex_codes = {"f": "Female", "female": "Female", "m": "Male", "male": "Male"}

activity["sex"] = activity["sex"].str.lower()
activity["sex"] = activity["sex"].replace(sex_codes)

activity["sex"].value_counts(dropna=False)

sex
Female    7
Male      5
<NA>      1
Name: count, dtype: Int64

The same idea works for `follow_up_needed`:

In [131]:
yes_no_codes = {"y": "Yes", "yes": "Yes", "n": "No", "no": "No"}

activity["follow_up_needed"] = activity["follow_up_needed"].str.lower()
activity["follow_up_needed"] = activity["follow_up_needed"].replace(yes_no_codes)

activity["follow_up_needed"].value_counts(dropna=False)

follow_up_needed
Yes     7
No      5
<NA>    1
Name: count, dtype: Int64

The blanks are still blank (`<NA>`). We will decide what to do with them in Step 7.

## Step 5: Turn number columns into real numbers

You cannot add up or average a column that pandas thinks is text. We need to convert it, but first we must see **why** it is text.

### 5a. Appointments

In [132]:
activity["appointments"].value_counts(dropna=False)

appointments
275              2
1,250            1
980              1
430              1
875              1
1 040            1
310              1
620              1
nine hundred     1
1280             1
1060             1
not available    1
Name: count, dtype: int64

Three problems: a comma inside a number (`1,250`), a number written as words, and the phrase `not available`.

We fix them in order:

1. Make it text and lower case, so the phrases are predictable.
2. Translate the words we recognise. `pd.NA` means *missing*: "not available" is not a number, so it becomes a blank.
3. Remove commas and spaces with the pattern `[,\s]` ("a comma or a space").

In [133]:
appointment_words = {"nine hundred": "900", "not available": pd.NA}

activity["appointments"] = activity["appointments"].astype("string").str.lower()
activity["appointments"] = activity["appointments"].replace(appointment_words)
activity["appointments"] = activity["appointments"].str.replace(r"[,\s]", "", regex=True)

activity["appointments"].value_counts(dropna=False)

appointments
275     2
1250    1
980     1
430     1
875     1
1040    1
310     1
620     1
900     1
1280    1
1060    1
<NA>    1
Name: count, dtype: Int64

Everything left now looks like a number, but it is still stored as text. `pd.to_numeric()` converts it.

- `errors="coerce"` means: if something still cannot be converted, make it missing instead of crashing.
- `.astype("Int64")` (capital **I**) stores whole numbers **and** allows missing values. Plain `int64` cannot hold blanks.

In [134]:
activity["appointments"] = pd.to_numeric(activity["appointments"], errors="coerce")
activity["appointments"] = activity["appointments"].astype("Int64")

activity["appointments"].dtype

Int64Dtype()

### 5b. Average waiting time

In [135]:
activity["average_wait_days"].value_counts(dropna=False)

average_wait_days
6         2
12        2
7 days    1
5         1
NaN       1
8 days    1
-1        1
9         1
10        1
7         1
11        1
Name: count, dtype: int64

Some values have the word `days` typed after them. Remove it, strip the leftover spaces, then convert. `Float64` (capital **F**) is the decimal type that allows missing values.

In [136]:
activity["average_wait_days"] = activity["average_wait_days"].astype("string")
activity["average_wait_days"] = activity["average_wait_days"].str.replace("days", "", regex=False)
activity["average_wait_days"] = activity["average_wait_days"].str.strip()
activity["average_wait_days"] = pd.to_numeric(activity["average_wait_days"], errors="coerce")
activity["average_wait_days"] = activity["average_wait_days"].astype("Float64")

activity["average_wait_days"]

0      7.0
1      5.0
2     <NA>
3      6.0
4      8.0
5     -1.0
6      9.0
7     10.0
8     12.0
9     12.0
10     6.0
11     7.0
12    11.0
Name: average_wait_days, dtype: Float64

### 5c. Impossible values

A waiting time cannot be negative. A negative number here is a typing mistake, so we treat it as unknown.

`.loc[rows, column]` picks out cells. The condition `activity["average_wait_days"] < 0` gives `True` for each row with a negative wait, so this line sets **only those cells** to missing.

In [137]:
negative_wait = activity["average_wait_days"] < 0
print("Negative waits found:", negative_wait.sum())

activity.loc[negative_wait, "average_wait_days"] = np.nan

Negative waits found: 1


### 5d. Reporting year

In [138]:
activity["reporting_year"] = pd.to_numeric(activity["reporting_year"], errors="coerce")
activity["reporting_year"] = activity["reporting_year"].astype("Int64")

activity.dtypes

area_code             string
area_name             string
reporting_year         Int64
age_group             string
sex                   string
appointments           Int64
average_wait_days    Float64
follow_up_needed      string
notes                 string
dtype: object

Every number column now has a number type. Compare this with the `object` columns you saw in the first look.

## Step 6: Remove duplicate rows

At the start, `.duplicated()` may have found fewer repeats than there really were, because `ch001` and `CH001` looked different. Now the text is consistent, more rows match exactly.

`keep=False` marks **every** copy, so we can see the repeats side by side before deleting anything:

In [139]:
activity[activity.duplicated(keep=False)]

,area_code,area_name,reporting_year,age_group,sex,appointments,average_wait_days,follow_up_needed,notes
8,CH003,<NA>,2024,65+,Female,275,12.0,Yes,area name missing
9,CH003,<NA>,2024,65+,Female,275,12.0,Yes,area name missing


In [140]:
rows_before = len(activity)
activity = activity.drop_duplicates()
rows_after = len(activity)

print("Rows before:", rows_before)
print("Rows after: ", rows_after)
print("Removed:    ", rows_before - rows_after)

Rows before: 13
Rows after:  12
Removed:     1


> **Why remove duplicates *after* fixing the text?** Try to explain it in one sentence.

## Step 7: Decide what to do with each missing value

There is no single rule. For each column, ask: *can I work out the true value, or would I be inventing it?*

In [141]:
activity.isna().sum()

area_code            0
area_name            1
reporting_year       0
age_group            0
sex                  1
appointments         1
average_wait_days    2
follow_up_needed     1
notes                5
dtype: int64

### 7a. Area name — work it out from other rows

One row has a blank area name, but its area code is known, and other rows with the same code do have a name. So the true name can be **looked up**, not guessed.

We build a lookup table: one area code → one area name. Step by step:

In [142]:
# Keep only the rows that have an area name
rows_with_name = activity.dropna(subset=["area_name"])

# Keep the first row for each area code
one_row_per_code = rows_with_name.drop_duplicates("area_code")

# Use the area code as the row label and keep just the name column
name_for_code = one_row_per_code.set_index("area_code")["area_name"]
name_for_code

area_code
CH001    Northside
CH002    Riverside
CH003     Hilltown
Name: area_name, dtype: string

`.map()` uses that lookup: for each row it takes the area code and returns the matching name. `.fillna()` then fills **only the blank cells** with those looked-up names; names that were already there are untouched.

In [143]:
looked_up_names = activity["area_code"].map(name_for_code)
activity["area_name"] = activity["area_name"].fillna(looked_up_names)

activity[["area_code", "area_name"]]

,area_code,area_name
0,CH001,Northside
1,CH001,Northside
2,CH001,Northside
3,CH002,Riverside
4,CH002,Riverside
5,CH002,Riverside
6,CH003,Hilltown
7,CH003,Hilltown
8,CH003,Hilltown
10,CH001,Northside


### 7b. Waiting time — fill with the median

The waiting times are missing (one was blank, one was the negative value we removed). Here we fill them with the **median**, the middle value when all waits are sorted. The median is used rather than the mean because one unusually long wait would pull the mean up but barely moves the median.

This is a judgement, not a fact: in a real report you would state that you did it and how many values it affected.

In [144]:
wait_median = activity["average_wait_days"].median()
print("Median wait:", wait_median)

activity["average_wait_days"] = activity["average_wait_days"].fillna(wait_median)

Median wait: 7.5


### 7c. Sex and follow-up — label as Unknown

We cannot look these up and must not guess them. Writing `Unknown` keeps the row and makes the gap visible in any count.

In [145]:
activity["sex"] = activity["sex"].fillna("Unknown")
activity["follow_up_needed"] = activity["follow_up_needed"].fillna("Unknown")

### 7d. Appointments — leave it missing

An appointment count is a record of real activity. Filling it with an average would put a number in the data that nobody ever counted. So it **stays blank**, and pandas will skip it when you later add up or average the column.

In [146]:
activity.isna().sum()

area_code            0
area_name            0
reporting_year       0
age_group            0
sex                  0
appointments         1
average_wait_days    0
follow_up_needed     0
notes                5
dtype: int64

The only blanks left are the appointment count we chose to keep and the `notes` column, where a blank simply means "no note".

## Step 8: Put the rows in a sensible order

`.sort_values()` sorts by one or more columns. `.reset_index(drop=True)` renumbers the rows 0, 1, 2… after sorting.

In [147]:
grain = ["area_code", "reporting_year", "age_group", "sex"]

activity = activity.sort_values(grain)
activity = activity.reset_index(drop=True)
activity

,area_code,area_name,reporting_year,age_group,sex,appointments,average_wait_days,follow_up_needed,notes
0,CH001,Northside,2024,0-17,Female,1250,7.0,Yes,school clinics
1,CH001,Northside,2024,18-64,Male,980,5.0,No,<NA>
2,CH001,Northside,2024,65+,Female,430,7.5,Yes,telephone follow-up
3,CH001,Northside,2025,0-17,Female,1280,6.0,No,<NA>
4,CH002,Riverside,2024,0-17,Female,875,6.0,No,<NA>
5,CH002,Riverside,2024,18-64,Male,1040,8.0,Yes,evening service
6,CH002,Riverside,2024,65+,Male,310,7.5,No,data-entry error in wait
7,CH002,Riverside,2025,18-64,Unknown,1060,7.0,Unknown,sex and follow-up missing
8,CH003,Hilltown,2024,0-17,Female,620,9.0,Yes,<NA>
9,CH003,Hilltown,2024,18-64,Male,900,10.0,No,<NA>


## Step 9: Save

`.to_csv()` writes the table to a file. `index=False` stops pandas from adding its row numbers as an extra column.

The file name says what is inside, so anyone opening the folder later knows what it is without opening it.

In [148]:
activity.to_csv("data/community_health_activity_clean.csv", index=False)
print("Saved data/community_health_activity_clean.csv:", activity.shape)

Saved data/community_health_activity_clean.csv: (12, 9)


---
# Part 2 — The area population file

The same ten steps, more quickly. Wherever you can, **predict the output before you run the cell**.

This file has a different grain:

> **one area × one year**

That matters later: several activity rows (one per age group and sex) will match the same population row.

### Look, then read

In [149]:
pd.read_csv(POPULATION_FILE, header=None, nrows=4)

,0,1,2,3,4,5
0,Area population and context extract - teaching...,NaN,NaN,NaN,NaN,NaN
1,area code,Area Name,Year,Population,REGION,Deprivation Band
2,CH001,Northside,2024,"52,400",North,Low
3,CH002,Riverside,2024,48750,north,medium


The same title line sits above the column names, so we skip it again. Then we reuse both functions we wrote earlier.

In [150]:
population_raw = pd.read_csv(POPULATION_FILE, skiprows=1)
quick_audit(population_raw, "Raw population data")

Raw population data
Rows and columns: (7, 6)
Duplicate rows: 1
Type and missing values per column:


,type,missing
area code,str,0
Area Name,str,0
Year,int64,0
Population,str,0
REGION,str,1
Deprivation Band,str,0


In [151]:
population = population_raw.copy()
population.columns = clean_headers(population.columns)
population.columns

Index(['area_code', 'area_name', 'year', 'population', 'region',
       'deprivation_band'],
      dtype='str')

### Fix the text

In [152]:
for column in ["area_code", "area_name", "region", "deprivation_band"]:
    population[column] = population[column].astype("string").str.strip()

population["area_code"] = population["area_code"].str.upper()
population["area_name"] = population["area_name"].str.title()
population["region"] = population["region"].str.title()
population["deprivation_band"] = population["deprivation_band"].str.title()

population["area_name"].value_counts(dropna=False)

area_name
Northside    2
Riverside    2
Hill Town    2
Hilltown     1
Name: count, dtype: Int64

One area is spelled two ways. Capitals and spaces cannot fix this, because the space is *inside* the name, so we replace it directly:

In [153]:
population["area_name"] = population["area_name"].replace({"Hill Town": "Hilltown"})
population["area_name"].value_counts(dropna=False)

area_name
Hilltown     3
Northside    2
Riverside    2
Name: count, dtype: Int64

### Fix the numbers

In [154]:
population["population"].value_counts(dropna=False)

population
not available    2
52,400           1
48750            1
36 900           1
53100            1
49,200           1
Name: count, dtype: int64

In [155]:
population["population"] = population["population"].astype("string").str.lower()
population["population"] = population["population"].replace({"not available": pd.NA})
population["population"] = population["population"].str.replace(r"[,\s]", "", regex=True)
population["population"] = pd.to_numeric(population["population"], errors="coerce").astype("Int64")

population["year"] = pd.to_numeric(population["year"], errors="coerce").astype("Int64")

population.dtypes

area_code           string
area_name           string
year                 Int64
population           Int64
region              string
deprivation_band    string
dtype: object

### Remove duplicates

In [156]:
rows_before = len(population)
population = population.drop_duplicates()
print("Removed:", rows_before - len(population))

Removed: 1


### Missing values

`region` has a blank. Like the area name in Part 1, it can be **looked up** from another row with the same area code: the same three-step lookup, then `.map()` and `.fillna()`.

`population` has a blank too. A population count is a real measurement, so, like the appointment count, it stays blank.

In [157]:
rows_with_region = population.dropna(subset=["region"])
one_row_per_code = rows_with_region.drop_duplicates("area_code")
region_for_code = one_row_per_code.set_index("area_code")["region"]

population["region"] = population["region"].fillna(population["area_code"].map(region_for_code))

population.isna().sum()

area_code           0
area_name           0
year                0
population          1
region              0
deprivation_band    0
dtype: int64

### Sort, check and save

In [158]:
population_grain = ["area_code", "year"]

population = population.sort_values(population_grain).reset_index(drop=True)

print("Rows sharing the same key:", population.duplicated(population_grain).sum())
print("Population blanks kept:   ", population["population"].isna().sum())


population

Rows sharing the same key: 0
Population blanks kept:    1


,area_code,area_name,year,population,region,deprivation_band
0,CH001,Northside,2024,52400,North,Low
1,CH001,Northside,2025,53100,North,Low
2,CH002,Riverside,2024,48750,North,Medium
3,CH002,Riverside,2025,49200,North,Medium
4,CH003,Hilltown,2024,36900,North,High
5,CH003,Hilltown,2025,<NA>,North,High


In [ ]:
population.to_csv("data/area_population_clean.csv", index=False)
print("Saved area_population_clean.csv:", population.shape)

Saved area_population_clean.csv: (6, 6)


---
## What you used today

| Tool | What it does |
|---|---|
| `pd.read_csv(file, header=None, nrows=4)` | peek at the raw lines |
| `pd.read_csv(file, skiprows=1)` | read, skipping a title line |
| `.shape`, `.dtypes`, `.isna().sum()`, `.duplicated().sum()` | the four first-look questions |
| `.value_counts(dropna=False)` | list every variant of a value, including blanks |
| `.str.strip()`, `.str.lower()`, `.str.upper()`, `.str.title()` | tidy text |
| `.str.replace(pattern, new, regex=True)` | replace text that matches a pattern |
| `.replace({old: new})` | translate whole values with a dictionary |
| `pd.to_numeric(..., errors="coerce")` | turn text into numbers, unreadable values become missing |
| `.astype("Int64")` / `.astype("Float64")` | number types that allow blanks |
| `.loc[condition, column] = np.nan` | blank out cells that break a rule |
| `.drop_duplicates()` | remove exact repeats |
| `.map(lookup)` + `.fillna()` | fill blanks from another row with the same key |
| `.fillna(value)` | fill blanks with a chosen value |
| `.to_csv(file, index=False)` | save |

## Exit ticket

Answer in a sentence or two each.

1. Which cleaning step changed how a value was **written** but not what it **meant**? Give one example.
2. Which missing values did you fill, and why was each fill reasonable?
3. Which missing values did you deliberately leave blank, and why?
4. What is the key (grain) of each clean file, and why are they different?